# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliHamza-lab/Ml_intership/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Two findings I would audit constructively are:

1. Finding: low CTR on page 1 is a useful warning sign for future decline. Methodology question: what is the label used to define “decline,” and does the validation design hold out whole clients or future time periods so the claim is not just measuring memorized client pattern? In other words, where does the label come from, and does the evaluation design actually support a future-decline claim?

2. Finding: the model’s ranking is more useful than raw accuracy because editors only review a small queue. Methodology question: does the paper report top-K hit rate against the base rate on a holdout that is blind to the outcome window, or does it rely mainly on ROC AUC from a random split that can overstate real-world usefulness? A careful reviewer would ask whether the claim is about rank quality or about general accuracy.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

frame = pd.read_csv('../../data/processed/refresh_feature_vector.csv')
print('rows', len(frame))
print('label mean', frame['is_declining_label'].mean())
print('columns including label and trend fields', [c for c in ['is_declining_label','trend_direction','trend_pct','avg_position','ctr'] if c in frame.columns])
# The notebook is written to be public-safe; it uses the processed file shipped in this repo.


rows 30000
label mean 0.5420666666666667
columns including label and trend fields ['is_declining_label', 'trend_direction', 'trend_pct', 'avg_position', 'ctr']


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

frame = pd.read_csv('../../data/processed/refresh_feature_vector.csv')

numeric_features = [
    'search_volume','competition','cpc','word_count','char_count',
    'log_impressions_90d','log_clicks_90d','log_sessions_90d','log_ai_sessions_90d',
    'days_with_impressions','days_with_sessions','content_age_days',
    'days_since_last_update','ctr','avg_position','engagement_rate','scroll_rate','ai_traffic_pct'
]
categorical_features = [
    'competition_level','content_type','main_intent','age_tier','freshness_tier',
    'word_count_tier','impression_tier','position_tier'
]

numeric_frame = frame[numeric_features].apply(pd.to_numeric, errors='coerce').fillna(0)
categorical_frame = frame[categorical_features].fillna('unknown').astype(str)
encoded_frame = pd.get_dummies(categorical_frame, prefix=categorical_features, dummy_na=False, dtype=float)
X = pd.concat([numeric_frame, encoded_frame], axis=1)
y = frame['is_declining_label'].astype(int)
clients = frame['client_id'].fillna('unknown').astype(str)

rng = np.random.default_rng(42)
unique_clients = np.array(clients.drop_duplicates())
rng.shuffle(unique_clients)
# 20% of clients held out as a grouped test set
n_test_clients = max(1, int(round(len(unique_clients) * 0.2)))
test_clients = set(unique_clients[:n_test_clients])
test_mask = clients.isin(test_clients).to_numpy()
train_idx = np.arange(len(frame))[~test_mask]
test_idx = np.arange(len(frame))[test_mask]

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Honest baseline from the repo's existing rule logic
baseline_lookup = pd.read_csv('../../data/processed/baseline_refresh_queue.csv').set_index('content_id')['baseline_refresh_score']
baseline_test_scores = frame.iloc[test_idx]['content_id'].map(baseline_lookup).fillna(0).to_numpy()

# A small helper for top-K hit rate

def precision_at_k(labels, scores, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return float(topk.mean()) if len(topk) else 0.0

models = {
    'logistic_regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
    ]),
    'decision_tree': DecisionTreeClassifier(class_weight='balanced', max_depth=5, min_samples_leaf=50, random_state=42),
    'random_forest': RandomForestClassifier(class_weight='balanced_subsample', max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=42),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]
    print(name, {
        'roc_auc': round(roc_auc_score(y_test, probs), 3),
        'average_precision': round(average_precision_score(y_test, probs), 3),
        'precision_at_20': round(precision_at_k(y_test, probs, 20), 3),
        'precision_at_50': round(precision_at_k(y_test, probs, 50), 3),
        'base_rate': round(y_test.mean(), 3),
    })

# Baseline comparison on the same held-out clients
print('baseline', {
    'roc_auc': round(roc_auc_score(y_test, baseline_test_scores), 3),
    'average_precision': round(average_precision_score(y_test, baseline_test_scores), 3),
    'precision_at_20': round(precision_at_k(y_test, baseline_test_scores, 20), 3),
    'precision_at_50': round(precision_at_k(y_test, baseline_test_scores, 50), 3),
    'base_rate': round(y_test.mean(), 3),
})


logistic_regression {'roc_auc': 0.7, 'average_precision': 0.522, 'precision_at_20': 0.35, 'precision_at_50': 0.4, 'base_rate': np.float64(0.391)}
decision_tree {'roc_auc': 0.742, 'average_precision': 0.575, 'precision_at_20': 0.8, 'precision_at_50': 0.68, 'base_rate': np.float64(0.391)}
random_forest {'roc_auc': 0.747, 'average_precision': 0.61, 'precision_at_20': 0.7, 'precision_at_50': 0.68, 'base_rate': np.float64(0.391)}
baseline {'roc_auc': 0.627, 'average_precision': 0.468, 'precision_at_20': 0.15, 'precision_at_50': 0.24, 'base_rate': np.float64(0.391)}


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

frame = pd.read_csv('../../data/processed/refresh_feature_vector.csv')

# Leakage audit: test whether a likely leaky feature changes the score materially.
# The matrix is kept small and memory-safe so the notebook can run on the repo's shipped data.

numeric_features = [
    'ctr', 'avg_position', 'days_with_impressions', 'days_since_last_update',
    'content_age_days', 'search_volume', 'engagement_rate', 'scroll_rate'
]

categorical_features = ['content_type', 'position_tier']

numeric_frame = frame[numeric_features].apply(pd.to_numeric, errors='coerce').fillna(0)
categorical_frame = frame[categorical_features].fillna('unknown').astype(str)
encoded_cats = categorical_frame.apply(lambda col: col.astype('category').cat.codes)

X = pd.concat([numeric_frame, encoded_cats], axis=1)
X['trend_pct'] = pd.to_numeric(frame['trend_pct'], errors='coerce').fillna(0)
y = frame['is_declining_label'].astype(int)

train_idx, test_idx = train_test_split(np.arange(len(frame)), test_size=0.2, random_state=42, stratify=y)
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

model = RandomForestClassifier(n_estimators=120, random_state=42, n_jobs=-1)
model.fit(X_train[['ctr', 'avg_position', 'days_with_impressions', 'days_since_last_update', 'content_age_days', 'search_volume', 'engagement_rate', 'scroll_rate', 'content_type', 'position_tier']], y_train)
probs = model.predict_proba(X_test[['ctr', 'avg_position', 'days_with_impressions', 'days_since_last_update', 'content_age_days', 'search_volume', 'engagement_rate', 'scroll_rate', 'content_type', 'position_tier']])[:, 1]
print('without trend_pct roc_auc', round(roc_auc_score(y_test, probs), 3))

X_leaky = X.copy()
X_leaky['trend_pct'] = frame['trend_pct'].astype(float)
model_leaky = RandomForestClassifier(n_estimators=120, random_state=42, n_jobs=-1)
model_leaky.fit(X_leaky.iloc[train_idx][['ctr', 'avg_position', 'days_with_impressions', 'days_since_last_update', 'content_age_days', 'search_volume', 'engagement_rate', 'scroll_rate', 'content_type', 'position_tier', 'trend_pct']], y_train)
probs_leaky = model_leaky.predict_proba(X_leaky.iloc[test_idx][['ctr', 'avg_position', 'days_with_impressions', 'days_since_last_update', 'content_age_days', 'search_volume', 'engagement_rate', 'scroll_rate', 'content_type', 'position_tier', 'trend_pct']])[:, 1]
print('with trend_pct roc_auc', round(roc_auc_score(y_test, probs_leaky), 3))
print('difference', round(roc_auc_score(y_test, probs_leaky) - roc_auc_score(y_test, probs), 3))


without trend_pct roc_auc 0.756
with trend_pct roc_auc 1.0
difference 0.244


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [5]:
import pandas as pd
import numpy as np

frame = pd.read_csv('../../data/processed/refresh_feature_vector.csv')
# Safe claim rewrite. Replace the boldest sentence in your report with a measured, decision-support version.
print('Observed and measured: the model ranker in this repo was evaluated on a grouped client holdout, with base rate', round(frame['is_declining_label'].mean(), 3))
print('Example rewrite: "In this repository, a client-held-out model produced directional ranking signals for refresh review, but it should be used as a decision-support aid rather than as proof of causality or a production guarantee."')


Observed and measured: the model ranker in this repo was evaluated on a grouped client holdout, with base rate 0.542
Example rewrite: "In this repository, a client-held-out model produced directional ranking signals for refresh review, but it should be used as a decision-support aid rather than as proof of causality or a production guarantee."


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
